In [ ]:
!pip install -q pypdf chromadb sentence-transformers langchain-text-splitters

In [ ]:
import os, shutil

os.makedirs("quickkart_corpus", exist_ok=True)

for name in os.listdir("."):
    if name.endswith((".txt", ".pdf")):
        shutil.move(name, os.path.join("quickkart_corpus", name))

print(sorted(os.listdir("quickkart_corpus")))

['cancellation_policy.txt', 'delivery_fees_and_slots.pdf', 'item_issues_policy.txt', 'late_delivery_policy.txt', 'payments_and_wallet.pdf', 'refund_policy.txt', 'special_cases_and_account_rules.pdf']


In [ ]:
from pypdf import PdfReader

os.makedirs("quickkart_corpus", exist_ok=True)
for name in os.listdir("."):
    if name.endswith((".txt", ".pdf")):
        shutil.move(name, os.path.join("quickkart_corpus", name))

folder = "quickkart_corpus"
for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)

    if name.endswith(".txt"):
        with open(path, encoding="utf-8") as f:
            text = f.read()
    elif name.endswith(".pdf"):
        text = ""
        for page in PdfReader(path).pages:
            text += page.extract_text()
    else:
        continue

    print(name, "->", len(text), "characters")
    print(text[:150])
    print("-----")

cancellation_policy.txt -> 2026 characters
QUICKKART CANCELLATION POLICY
Version 2.4 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real c
-----
delivery_fees_and_slots.pdf -> 1279 characters
QUICKKART DELIVERY FEES AND SLOTS
Version 3.1 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a re
-----
item_issues_policy.txt -> 1979 characters
QUICKKART ITEM ISSUES POLICY
Version 1.9 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real co
-----
late_delivery_policy.txt -> 2451 characters
QUICKKART LATE DELIVERY POLICY
Version 2.7 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real 
-----
payments_and_wallet.pdf -> 1961 characters
QUICKKART PAYMENTS AND WALLET
Version 4.0 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real c
--

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = {}
for name in sorted(os.listdir("quickkart_corpus")):
    path = os.path.join("quickkart_corpus", name)
    if name.endswith(".txt"):
        with open(path, encoding="utf-8") as f:
            docs[name] = f.read()
    elif name.endswith(".pdf"):
        docs[name] = "\n".join(page.extract_text() for page in PdfReader(path).pages)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = []
for name, text in docs.items():
    for piece in splitter.split_text(text):
        chunks.append({"text": piece, "source": name})

print("Total chunks:", len(chunks))
for name in docs:
    count = sum(1 for c in chunks if c["source"] == name)
    print(name, "->", count, "chunks")

print()
print(chunks[10])

Total chunks: 42
cancellation_policy.txt -> 6 chunks
delivery_fees_and_slots.pdf -> 3 chunks
item_issues_policy.txt -> 6 chunks
late_delivery_policy.txt -> 7 chunks
payments_and_wallet.pdf -> 6 chunks
refund_policy.txt -> 10 chunks
special_cases_and_account_rules.pdf -> 4 chunks

{'text': 'SECTION 1: TYPES OF ITEM ISSUES\n1.1 Missing item: an item you ordered and paid for is not in the delivery.\n1.2 Wrong item: you received a different product, variant or size from the one you ordered.\n1.3 Damaged item: the item or its packaging is broken, crushed or leaking.\n1.4 Spoiled or poor-quality item: for example rotten fruit, sour milk or wilted vegetables.', 'source': 'item_issues_policy.txt'}


In [ ]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("quickkart")

collection.add(
    ids=[f"chunk{i}" for i in range(len(chunks))],
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
)

print("Stored:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 61.3MiB/s]


Stored: 42


In [ ]:
def ask(question, n=3):
    results = collection.query(query_texts=[question], n_results=n)
    for i in range(n):
        source = results["metadatas"][0][i]["source"]
        distance = results["distances"][0][i]
        print(f"#{i+1} | {source} | distance {distance:.3f}")
        print(results["documents"][0][i])
        print()

ask("I received a yogurt that was already past its expiry date. What do I get and what do I need to do?")

#1 | item_issues_policy.txt | distance 1.162
SECTION 3: EVIDENCE REQUIRED
3.1 Missing items: no photo is needed. QuickKart checks the packing record instead.
3.2 Wrong, damaged and spoiled items: one clear photo of the item and its packaging or label is required.
3.3 Expired items: send a photo that clearly shows the expiry date on the label.
3.4 If a photo is unclear, support may ask for one more photo within 12 hours of your report.

#2 | item_issues_policy.txt | distance 1.252
1.5 Expired item: an item delivered after the expiry date printed on its label. You receive a full refund plus a Rs. 50 QuickKart Wallet credit.

#3 | refund_policy.txt | distance 1.284
SECTION 4: RETURNS AND NON-REFUNDABLE SITUATIONS
4.1 Change of mind: perishable products (fruits, vegetables, dairy, bakery, meat and frozen food) cannot be returned or refunded because you changed your mind. Sealed, unused non-perishable products in original packaging can be returned within 7 days of delivery. A return pickup 

In [ ]:
import pandas as pd

key = pd.read_csv("answer_key.csv")
key = key[key["source_file"] != "NONE"]

key.loc[key["id"] == 13, "must_contain"] += "|Perishable items: report within 24 hours"

def found(chunk_texts, needed):
    text = " ".join(" ".join(chunk_texts).split())
    return all(phrase in text for phrase in needed)

def evaluate(k=3):
    top1 = topk = 0
    for _, row in key.iterrows():
        results = collection.query(query_texts=[row["question"]], n_results=k)
        retrieved = results["documents"][0]
        needed = row["must_contain"].split("|")
        top1 += found(retrieved[:1], needed)
        topk += found(retrieved, needed)
        if not found(retrieved, needed):
            print("MISS in top", k, ": Q", row["id"])
    print(f"Top-1: {top1}/{len(key)}   Top-{k}: {topk}/{len(key)}")

evaluate()

MISS in top 3 : Q 13
MISS in top 3 : Q 17
Top-1: 9/17   Top-3: 15/17


In [ ]:
def build_collection(chunk_size, overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    pieces = []
    for name, text in docs.items():
        for p in splitter.split_text(text):
            pieces.append({"text": p, "source": name})
    col = client.get_or_create_collection(f"quickkart_{chunk_size}_{overlap}")
    if col.count() == 0:
        col.add(
            ids=[f"chunk{i}" for i in range(len(pieces))],
            documents=[p["text"] for p in pieces],
            metadatas=[{"source": p["source"]} for p in pieces],
        )
    return col, len(pieces)

def evaluate(col, k=3):
    top1 = topk = 0
    missed = []
    for _, row in key.iterrows():
        results = col.query(query_texts=[row["question"]], n_results=k)
        retrieved = results["documents"][0]
        needed = row["must_contain"].split("|")
        top1 += found(retrieved[:1], needed)
        ok = found(retrieved, needed)
        topk += ok
        if not ok:
            missed.append(int(row["id"]))
    return top1, topk, missed

for size, overlap in [(200, 20), (200, 50), (500, 50), (500, 100), (1000, 100)]:
    col, n = build_collection(size, overlap)
    t1, tk, missed = evaluate(col)
    print(f"size={size} overlap={overlap} chunks={n} | top-1 {t1}/{len(key)} | top-3 {tk}/{len(key)} | missed {missed}")

size=200 overlap=20 chunks=106 | top-1 6/17 | top-3 10/17 | missed [2, 3, 6, 7, 9, 10, 13]
size=200 overlap=50 chunks=108 | top-1 6/17 | top-3 11/17 | missed [3, 6, 7, 9, 10, 13]
size=500 overlap=50 chunks=42 | top-1 9/17 | top-3 15/17 | missed [13, 17]
size=500 overlap=100 chunks=43 | top-1 8/17 | top-3 16/17 | missed [13]
size=1000 overlap=100 chunks=22 | top-1 10/17 | top-3 16/17 | missed [8]


In [ ]:
def show(col_name, qid, k=3):
    col = client.get_collection(col_name)
    row = key[key["id"] == qid].iloc[0]
    print("Q", qid, "|", row["question"])
    print("Needs:", row["must_contain"])
    print()
    res = col.query(query_texts=[row["question"]], n_results=k)
    for i in range(k):
        print(f"#{i+1} | {res['metadatas'][0][i]['source']} | distance {res['distances'][0][i]:.3f}")
        print(res["documents"][0][i])
        print()

show("quickkart_500_100", 13)
show("quickkart_1000_100", 8)

Q 13 | I received a yogurt that was already past its expiry date. What do I get and what do I need to do?
Needs: full refund plus a Rs. 50|expiry date on the label|Perishable items: report within 24 hours

#1 | item_issues_policy.txt | distance 0.990
1.4 Spoiled or poor-quality item: for example rotten fruit, sour milk or wilted vegetables.
1.5 Expired item: an item delivered after the expiry date printed on its label. You receive a full refund plus a Rs. 50 QuickKart Wallet credit.

#2 | item_issues_policy.txt | distance 1.162
SECTION 3: EVIDENCE REQUIRED
3.1 Missing items: no photo is needed. QuickKart checks the packing record instead.
3.2 Wrong, damaged and spoiled items: one clear photo of the item and its packaging or label is required.
3.3 Expired items: send a photo that clearly shows the expiry date on the label.
3.4 If a photo is unclear, support may ask for one more photo within 12 hours of your report.

#3 | refund_policy.txt | distance 1.284
SECTION 4: RETURNS AND NON-REFU

In [ ]:
!pip install -q groq

In [ ]:
from groq import Groq
from google.colab import userdata
import time

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-20b"

def answer(question, k=5):
    res = col.query(query_texts=[question], n_results=k)
    context = ""
    for text, meta in zip(res["documents"][0], res["metadatas"][0]):
        context += f"[Source: {meta['source']}]\n{text}\n\n"

    prompt = f"""You are a customer support assistant for QuickKart.
Answer the question using ONLY the policy excerpts below.
- Mention the source file name(s) you used.
- If the excerpts do not contain the answer, reply exactly: "I don't know based on the policy documents."
- Do not guess or use outside knowledge.
- If a rule depends on a time, amount or condition, compare it with the details in the question and say which case applies, with the number for that case.
- If the answer depends on a condition the question does not mention, state that condition.

POLICY EXCERPTS:
{context}
QUESTION: {question}"""

    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e):
                return "RATE LIMITED: " + str(e)[:200]
            print(f"Attempt {attempt+1} failed: {str(e)[:80]}. Waiting 5 seconds...")
            time.sleep(5)
    return "FAILED after 3 attempts"

print(answer("One item was missing from my order. How much refund do I get?"))

You will receive a **full refund of the price of the missing item, including the tax that was charged on that item**.  

*Source: refund_policy.txt*


In [ ]:
print(answer("I cancelled my order 10 minutes after placing it, before the delivery partner picked it up. What will I pay?"))
print("-----")
print(answer("I received a yogurt that was already past its expiry date. What do I get and what do I need to do?"))
print("-----")
print(answer("Can I change my delivery address after I have placed an order?"))
print("-----")
print(answer("Does QuickKart sell alcohol or cigarettes?"))

You will be charged a cancellation fee of **Rs. 20** (deducted from your refund).  
*Source: cancellation_policy.txt – Section 1.2*
-----
**What you’ll receive**  
- A full refund of the yogurt’s price (including tax) **plus** a Rs. 50 QuickKart Wallet credit.  
  *Source: item_issues_policy.txt – Section 1.5; refund_policy.txt – Section 2.1, 2.3.*

**What you need to do**  
1. **Report the issue within 24 hours** of delivery (yogurt is a perishable item).  
   *Source: item_issues_policy.txt – Section 2.1.*  
2. **Send a clear photo** that shows the expiry date on the yogurt’s label.  
   *Source: item_issues_policy.txt – Section 3.3.*  

Once you submit the report with the photo within the 24‑hour window, QuickKart will process the refund and credit.
-----
I don't know based on the policy documents.
-----
I don't know based on the policy documents.


In [ ]:
import time
import pandas as pd

key_df = pd.read_csv("answer_key.csv")
results = []

for _, row in key_df.iterrows():
    reply = answer(row["question"])
    if reply.startswith("RATE LIMITED"):
        print("Stopped at question", row["id"], "-", reply)
        break
    results.append({
        "id": row["id"],
        "question": row["question"],
        "type": row["type"],
        "expected": row["expected_answer"],
        "model": MODEL,
        "model_answer": reply,
    })
    time.sleep(4)

df = pd.DataFrame(results)
df.to_csv("generation_results_v2.csv", index=False)

for _, r in df.iterrows():
    print("Q", r["id"], "|", r["question"])
    print("EXPECTED:", r["expected"])
    print("MODEL   :", r["model_answer"])
    print("-----")

Q 1 | One item was missing from my order. How much refund do I get?
EXPECTED: A full refund of the missing item's price, including the tax charged on that item.
MODEL   : You will receive a **full refund of the price of the missing item, including the tax that was charged on that item**.  

*Source: refund_policy.txt*
-----
Q 2 | I paid by UPI. How long will my refund take?
EXPECTED: 2 to 3 working days (working days = Monday to Saturday, excluding public holidays).
MODEL   : The refund will be processed within **2 to 3 working days** for a UPI payment.  
(See *refund_policy.txt*, Section 3.2.)
-----
Q 3 | I paid by credit card. How long will the refund take, and is there a faster option?
EXPECTED: 5 to 7 working days to the card. Faster option: choose QuickKart Wallet credit, which arrives within 5 minutes, but the choice cannot be reversed.
MODEL   : **Refund timeline for a credit‑card payment**  
- According to the Refund Policy (Section 3.2 of *refund_policy.txt*), a refund to a cr